In [16]:
from transformer_architecture import Transformer

import torch
import pandas as pd
import numpy as np
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(torch.__version__)
print(DEVICE)

2.5.1
cuda


In [17]:
SEQ_LEN = 10
VAL_FRAC = 10
TEST_FRAC = 10

In [18]:
df = pd.read_csv("cleaned_dataset.csv")
df.head()

,Entity,Year,Access to electricity (% of population),Access to clean fuels for cooking,Renewable-electricity-generating-capacity-per-capita,Renewable energy share in the total final energy consumption (%),Electricity from fossil fuels (TWh),Electricity from nuclear (TWh),Electricity from renewables (TWh),Low-carbon electricity (% electricity),Primary energy consumption per capita (kWh/person),Energy intensity level of primary energy (MJ/$2017 PPP GDP),Value_co2_emissions_kt_by_country,gdp_per_capita,Density\n(P/Km2),Land Area(Km2),Latitude,Longitude
0,Algeria,2000,98.97310,97.1,8.91,0.43,23.84,0.0,0.05,0.209293,9746.524,4.18,80050.00000,1765.027146,18.0,2381741.0,28.033886,1.659626
1,Algeria,2001,98.96687,97.3,8.79,0.43,24.96,0.0,0.07,0.279664,9961.640,4.07,78650.00000,1740.606654,18.0,2381741.0,28.033886,1.659626
2,Algeria,2002,98.95306,97.8,8.68,0.51,25.94,0.0,0.06,0.230769,10180.350,4.12,82400.00153,1781.828908,18.0,2381741.0,28.033886,1.659626
3,Algeria,2003,98.93401,98.0,8.57,0.47,27.54,0.0,0.26,0.935252,10510.461,4.08,88190.00244,2103.381291,18.0,2381741.0,28.033886,1.659626
4,Algeria,2004,98.91208,98.2,8.46,0.44,29.14,0.0,0.25,0.850630,10759.022,3.96,89489.99786,2610.185422,18.0,2381741.0,28.033886,1.659626


In [19]:
df[df["Entity"] == "Somalia"]

,Entity,Year,Access to electricity (% of population),Access to clean fuels for cooking,Renewable-electricity-generating-capacity-per-capita,Renewable energy share in the total final energy consumption (%),Electricity from fossil fuels (TWh),Electricity from nuclear (TWh),Electricity from renewables (TWh),Low-carbon electricity (% electricity),Primary energy consumption per capita (kWh/person),Energy intensity level of primary energy (MJ/$2017 PPP GDP),Value_co2_emissions_kt_by_country,gdp_per_capita,Density\n(P/Km2),Land Area(Km2),Latitude,Longitude


In [20]:
df_global = df.drop(columns=["Entity"]).groupby('Year', as_index=False).mean()
df_global.head()

,Year,Access to electricity (% of population),Access to clean fuels for cooking,Renewable-electricity-generating-capacity-per-capita,Renewable energy share in the total final energy consumption (%),Electricity from fossil fuels (TWh),Electricity from nuclear (TWh),Electricity from renewables (TWh),Low-carbon electricity (% electricity),Primary energy consumption per capita (kWh/person),Energy intensity level of primary energy (MJ/$2017 PPP GDP),Value_co2_emissions_kt_by_country,gdp_per_capita,Density\n(P/Km2),Land Area(Km2),Latitude,Longitude
0,2000,69.807113,57.502518,63.531727,37.933669,56.533094,15.789137,17.499496,39.827553,24855.480132,6.338345,135961.942446,7182.413910,122.92118,668916.71223,17.830211,11.59866
1,2001,70.387125,58.055036,63.405180,37.434820,57.557482,16.222518,16.999496,39.027987,24931.953828,6.259928,138294.604317,7087.544674,122.92118,668916.71223,17.830211,11.59866
2,2002,71.212831,58.609353,64.333741,37.383165,60.198201,16.245612,17.532590,39.176419,25414.709272,6.132230,140026.257602,7493.740220,122.92118,668916.71223,17.830211,11.59866
3,2003,71.838035,59.178058,64.689137,37.007338,63.443022,15.924101,17.708489,38.727300,25886.024066,6.112014,146716.405106,8786.377507,122.92118,668916.71223,17.830211,11.59866
4,2004,72.447574,59.713669,64.607842,36.858777,66.113094,16.700791,19.046906,38.993923,26629.698052,5.968201,153865.898509,10099.216825,122.92118,668916.71223,17.830211,11.59866


## Entity - Year Pair

In [21]:
entity = df["Entity"].unique()
year = df["Year"].unique()
year_start = np.arange(year.min(), year.max() - SEQ_LEN + 1, 1)
print(year_start)
len(entity), len(year), len(year_start)

[2000 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010]


(139, 21, 11)

In [22]:
ds = []

for e in entity:
    for y in year_start:
        ds.append((e, y))

len(ds)

1529

# Preparation

In [23]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

In [24]:
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-3

HIDDEN_DIM = 64

In [25]:
temp_size = VAL_FRAC + TEST_FRAC
ds_train, ds_temp = train_test_split(ds, test_size= temp_size/100)
ds_val, ds_test = train_test_split(ds_temp, train_size = VAL_FRAC/temp_size)

len(ds_train), len(ds_val), len(ds_test)

(1223, 153, 153)

In [26]:
train_loader = DataLoader(ds_train, batch_size = BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ds_val, batch_size = BATCH_SIZE)
test_loader = DataLoader(ds_test, batch_size = BATCH_SIZE)

In [27]:
col_count = len(df_global.columns)

model = Transformer(
    hidden_dim=HIDDEN_DIM,
    seq_len=10,
    corpus_size=col_count,
    encoder_attention_heads=4,
    encoder_blocks=2,
    decoder_attention_heads=4,
    decoder_blocks=2,
    use_embedding=False,
    embedding_replacement=torch.nn.Linear(col_count, HIDDEN_DIM)
).to(DEVICE)

criterion = torch.nn.SmoothL1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

In [28]:
df = df.sort_values(["Entity", "Year"]).set_index(["Entity", "Year"], drop=False)
df = df.drop(columns=["Entity"])
df_global = df_global.sort_values("Year").set_index("Year", drop=False)

entity_groups = {
    k: v.values for k, v in df.groupby(level=0)
}
global_array = df_global.values
years_index = {k: v for v, k in enumerate(df_global.index.to_numpy())}
years_index

{np.int64(2000): 0,
 np.int64(2001): 1,
 np.int64(2002): 2,
 np.int64(2003): 3,
 np.int64(2004): 4,
 np.int64(2005): 5,
 np.int64(2006): 6,
 np.int64(2007): 7,
 np.int64(2008): 8,
 np.int64(2009): 9,
 np.int64(2010): 10,
 np.int64(2011): 11,
 np.int64(2012): 12,
 np.int64(2013): 13,
 np.int64(2014): 14,
 np.int64(2015): 15,
 np.int64(2016): 16,
 np.int64(2017): 17,
 np.int64(2018): 18,
 np.int64(2019): 19,
 np.int64(2020): 20}

# Training Loop

In [29]:
def load_tensor(entities, years):
    regional_data = []
    global_data = []

    for entity, year in zip(entities, years):
        year = int(year)
        entity = entity.item() if hasattr(entity, "item") else entity

        regional_data.append(torch.tensor(entity_groups[entity][years_index[year] : years_index[year + SEQ_LEN] + 1]))
        global_data.append(torch.tensor(global_array[years_index[year] : years_index[year + SEQ_LEN]]))


    regional_data = torch.stack(regional_data).to(DEVICE, torch.float32)
    global_data = torch.stack(global_data).to(DEVICE, torch.float32)
    return regional_data, global_data

In [ ]:
for e in range(EPOCHS):
    model.train()
    train_loss = 0
    for entities, years in tqdm(train_loader, desc=f"Epoch {e:>3d} Training"):
        regional_data, global_data = load_tensor(entities, years)
        pred = model(global_data, regional_data[:, :-1])
        ground_truth = regional_data[:, 1:]

        loss = criterion(pred, ground_truth)
        train_loss += loss

        loss.backward()
        optimizer.step()


Epoch   8 Training, loss: 0, progress:  10%|█         | 4/39 [00:00<00:01, 19.55it/s]


KeyboardInterrupt: 